# Step 5 — Automate ML Pipeline

En este notebook se valida la automatización del pipeline de Machine Learning con DVC.

El objetivo es confirmar que el flujo puede ejecutarse completo mediante:

dvc repro

y también forzar una reproducción completa mediante:

dvc repro -f

Esto permite validar que el proyecto es reproducible sin ejecutar manualmente cada script.

# Imports

In [3]:
# IMPORTS

from pathlib import Path
import subprocess
import json
import yaml

# Config

In [5]:
# CONFIG

PROJECT_ROOT = Path.cwd()

PARAMS_PATH = PROJECT_ROOT / "params.yaml"
DVC_YAML_PATH = PROJECT_ROOT / "dvc.yaml"
DVC_LOCK_PATH = PROJECT_ROOT / "dvc.lock"

with open(PARAMS_PATH, "r", encoding="utf-8") as f:
    params = yaml.safe_load(f)

print("Project root:", PROJECT_ROOT)
print("params.yaml exists:", PARAMS_PATH.exists())
print("dvc.yaml exists:", DVC_YAML_PATH.exists())
print("dvc.lock exists:", DVC_LOCK_PATH.exists())

Project root: C:\Users\daniel.martinez\real_state_price_predictor\tog_dme
params.yaml exists: True
dvc.yaml exists: True
dvc.lock exists: True


## 1. Helper para comandos

In [7]:
# HELPER FUNCTION

def run_command(command):
    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        shell=True
    )

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print("STDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Falló el comando: {command}")

    return result

## 1. Revisar estado inicial del pipeline

In [9]:
# DVC STATUS

run_command("dvc status")

Data and pipelines are up to date.



CompletedProcess(args='dvc status', returncode=0, stdout='Data and pipelines are up to date.\n', stderr='')

## 2. Reproducir pipeline

In [11]:
# DVC REPRO

run_command("dvc repro")

Stage 'prepare_data' didn't change, skipping
Stage 'split_data' didn't change, skipping
Stage 'train' didn't change, skipping
Stage 'evaluate' didn't change, skipping
Data and pipelines are up to date.



CompletedProcess(args='dvc repro', returncode=0, stdout="Stage 'prepare_data' didn't change, skipping\nStage 'split_data' didn't change, skipping\nStage 'train' didn't change, skipping\nStage 'evaluate' didn't change, skipping\nData and pipelines are up to date.\n", stderr='')

## 3. Forzar reproducción completa

In [13]:
# FORCE DVC REPRO

run_command("dvc repro -f")

Running stage 'prepare_data':
> python src/stages/prepare_data.py
Prepared data saved to: data/processed/prepared_data.csv
Prepared data shape: (536, 18)

Running stage 'split_data':
> python src/stages/split_data.py
Prepared data loaded from: data/processed/prepared_data.csv
Prepared data shape: (536, 18)
Train data saved to: data/processed/train_data.csv
Test data saved to: data/processed/test_data.csv
Train shape: (428, 18)
Test shape: (108, 18)
Scaler X saved to: models/scaler_X.pkl
Scaler Y saved to: models/scaler_Y.pkl
Feature names saved to: models/feature_names.json

Running stage 'train':
> python src/stages/train.py
Train data loaded from: data/processed/train_data.csv
Train data shape: (428, 18)
Modelo entrenado correctamente.
Mejores hiperparámetros: {'alpha': 20}
R2 CV train: 0.8533903044819209
Modelo guardado en: models/modelo_final.pkl

Running stage 'evaluate':
> python src/stages/evaluate.py
Train data loaded from: data/processed/train_data.csv
Test data loaded from: d

CompletedProcess(args='dvc repro -f', returncode=0, stdout="Running stage 'prepare_data':\n> python src/stages/prepare_data.py\nPrepared data saved to: data/processed/prepared_data.csv\nPrepared data shape: (536, 18)\n\nRunning stage 'split_data':\n> python src/stages/split_data.py\nPrepared data loaded from: data/processed/prepared_data.csv\nPrepared data shape: (536, 18)\nTrain data saved to: data/processed/train_data.csv\nTest data saved to: data/processed/test_data.csv\nTrain shape: (428, 18)\nTest shape: (108, 18)\nScaler X saved to: models/scaler_X.pkl\nScaler Y saved to: models/scaler_Y.pkl\nFeature names saved to: models/feature_names.json\n\nRunning stage 'train':\n> python src/stages/train.py\nTrain data loaded from: data/processed/train_data.csv\nTrain data shape: (428, 18)\nModelo entrenado correctamente.\nMejores hiperparámetros: {'alpha': 20}\nR2 CV train: 0.8533903044819209\nModelo guardado en: models/modelo_final.pkl\n\nRunning stage 'evaluate':\n> python src/stages/eva

## 5. Visualizar DAG del pipeline

In [15]:
# DVC DAG

run_command("dvc dag")

       +--------------+    
       | prepare_data |    
       +--------------+    
               *           
               *           
               *           
        +------------+     
        | split_data |     
        +------------+     
         **        **      
       **            **    
      *                **  
+-------+                * 
| train |              **  
+-------+            **    
         **        **      
           **    **        
             *  *          
         +----------+      
         | evaluate |      
         +----------+      



CompletedProcess(args='dvc dag', returncode=0, stdout='       +--------------+    \n       | prepare_data |    \n       +--------------+    \n               *           \n               *           \n               *           \n        +------------+     \n        | split_data |     \n        +------------+     \n         **        **      \n       **            **    \n      *                **  \n+-------+                * \n| train |              **  \n+-------+            **    \n         **        **      \n           **    **        \n             *  *          \n         +----------+      \n         | evaluate |      \n         +----------+      \n', stderr='')

## 6. Mostrar métricas con DVC

In [17]:
# DVC METRICS SHOW

run_command("dvc metrics show")

Path                  MAE_test_pesos    MAE_train_pesos    R2_test    R2_train    RMSE_test_pesos    RMSE_train_pesos
reports\metrics.json  549476.60932      598614.59216       0.85867    0.87228     724228.72872       820764.7608



CompletedProcess(args='dvc metrics show', returncode=0, stdout='Path                  MAE_test_pesos    MAE_train_pesos    R2_test    R2_train    RMSE_test_pesos    RMSE_train_pesos\nreports\\metrics.json  549476.60932      598614.59216       0.85867    0.87228     724228.72872       820764.7608\n', stderr='')

## 7. Validar métricas contra fase 1

In [19]:
# LOAD METRICS

metrics_path = PROJECT_ROOT / params["metrics"]["metrics_path"]

with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

metrics

{'R2_train': 0.8722779520936734,
 'R2_test': 0.8586681107871221,
 'RMSE_train_pesos': 820764.7607973475,
 'RMSE_test_pesos': 724228.728716661,
 'MAE_train_pesos': 598614.5921648004,
 'MAE_test_pesos': 549476.6093178215}

In [20]:
# COMPARE AGAINST PHASE 1

expected_metrics = {
    "R2_test": 0.8586681107871221,
    "RMSE_test_pesos": 724228.7287166608,
    "MAE_test_pesos": 549476.6093178215,
}

for metric_name, expected_value in expected_metrics.items():
    current_value = metrics[metric_name]
    difference = abs(current_value - expected_value)

    print(f"{metric_name}")
    print(f"  Esperado: {expected_value}")
    print(f"  Actual:   {current_value}")
    print(f"  Dif:      {difference}")
    print()

assert abs(metrics["R2_test"] - expected_metrics["R2_test"]) < 1e-8
assert abs(metrics["RMSE_test_pesos"] - expected_metrics["RMSE_test_pesos"]) < 1e-2
assert abs(metrics["MAE_test_pesos"] - expected_metrics["MAE_test_pesos"]) < 1e-2

print("Pipeline automatizado validado correctamente.")

R2_test
  Esperado: 0.8586681107871221
  Actual:   0.8586681107871221
  Dif:      0.0

RMSE_test_pesos
  Esperado: 724228.7287166608
  Actual:   724228.728716661
  Dif:      2.3283064365386963e-10

MAE_test_pesos
  Esperado: 549476.6093178215
  Actual:   549476.6093178215
  Dif:      0.0

Pipeline automatizado validado correctamente.


## Conclusión

El pipeline puede reproducirse automáticamente con DVC usando:

dvc repro

Además, puede forzarse la ejecución completa con:

dvc repro -f

Esto confirma que el flujo de datos, entrenamiento y evaluación ya está automatizado y mantiene las mismas métricas del prototipo original.

In [22]:
# COMPARE AGAINST PHASE 1 RESULTS

expected_metrics = {
    "R2_test": 0.8586681107871221,
    "RMSE_test_pesos": 724228.7287166608,
    "MAE_test_pesos": 549476.6093178215,
}

print("Comparación contra monolito de Fase 1:")
print("-" * 60)

for metric_name, expected_value in expected_metrics.items():
    current_value = metrics[metric_name]
    difference = abs(current_value - expected_value)

    print(f"{metric_name}")
    print(f"  Esperado: {expected_value}")
    print(f"  Actual:   {current_value}")
    print(f"  Dif:      {difference}")
    print()

assert abs(metrics["R2_test"] - expected_metrics["R2_test"]) < 1e-8
assert abs(metrics["RMSE_test_pesos"] - expected_metrics["RMSE_test_pesos"]) < 1e-2
assert abs(metrics["MAE_test_pesos"] - expected_metrics["MAE_test_pesos"]) < 1e-2

print("Métricas validadas correctamente contra Fase 1.")

Comparación contra monolito de Fase 1:
------------------------------------------------------------
R2_test
  Esperado: 0.8586681107871221
  Actual:   0.8586681107871221
  Dif:      0.0

RMSE_test_pesos
  Esperado: 724228.7287166608
  Actual:   724228.728716661
  Dif:      2.3283064365386963e-10

MAE_test_pesos
  Esperado: 549476.6093178215
  Actual:   549476.6093178215
  Dif:      0.0

Métricas validadas correctamente contra Fase 1.


## Conclusión

El pipeline DVC reproduce correctamente el flujo completo del modelo.

Con este paso, el proyecto deja de depender de ejecutar scripts manualmente y puede reproducirse mediante:

dvc repro

Esto confirma que el modelo ganador de Fase 1 fue industrializado como pipeline reproducible.ducible.